# Learning Interference

**Track:** Learning
**Construct:** Rule induction under competing interference

Tests whether LLMs can induce transformation rules from input-output examples and apply them correctly despite interference from competing rule systems.

## Cognitive Science Background

When multiple similar rule systems are presented together, **proactive and retroactive interference** degrade learning (Underwood, 1957). The **Einstellung effect** (Luchins, 1942) shows that priming with a particular solution method biases subsequent problem-solving even when that method is incorrect. This benchmark tests rule induction — rules are never stated as text, only demonstrated through examples — under increasing interference.

## Methodology

Four tiers of increasing interference challenge the model's ability to induce and apply rules:


| Tier | Weight | Description |
|------|--------|-------------|
| Clean Induction | 0.10 | 5 examples from one system — baseline induction ability |
| Labeled Groups | 0.25 | Two labeled groups with same symbols but different rules |
| Interleaved + Anti-Pattern | 0.35 | 3 interleaved systems with Einstellung-effect priming |
| Unlabeled Clustering | 0.30 | 4 systems, 12 unlabeled examples — must cluster and induce |


## Scoring

$$\text{Score} = 0.10 \\text{times} \text{Tier1} + 0.25 \\text{times} \text{Tier2} + 0.35 \\text{times} \text{Tier3} + 0.30 \\text{times} \text{Tier4}$$

| Score | Interpretation |
|:---:|---|
| 0.8–1.0 | Robust to interference — separates interleaved rule systems and resists priming |
| 0.5–0.8 | Good — handles labeled separation but struggles with unlabeled clustering |
| 0.2–0.5 | Moderate — clean induction works but cross-contamination degrades accuracy |
| 0.0–0.2 | Severe interference — cannot separate concurrent rule systems |

### References

Underwood (1957), Luchins (1942)

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null
import kaggle_benchmarks as kbench

In [ ]:
import kaggle_benchmarks as kbench

"""
Novel Rule System Generator for Learning Benchmarks.

Generates procedural rule systems that cannot be in training data.
Each system defines a mapping from inputs to outputs via a chain
of deterministic rules. Difficulty is controlled by:
- Number of rules
- Number of input features
- Rule interaction complexity (independent vs. chained)

Systems are seeded for reproducibility across runs.
"""

import random
import hashlib
import copy
from dataclasses import dataclass, field


@dataclass
class RuleSystem:
    """A generated rule system with examples."""
    name: str
    description: str
    rules: list[str]
    examples: list[dict]  # {"input": str, "output": str}
    test_items: list[dict]  # {"input": str, "output": str}
    difficulty: int  # 1-3
    n_rules: int
    domain: str  # "symbol", "language", "number"


def _make_rng(seed: str) -> random.Random:
    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)
    return random.Random(h)


def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a symbol transformation rule system.

    Input: sequence of symbols (e.g., "△ ○ □")
    Rules: transformations (e.g., "△ followed by ○ becomes ★")
    Output: transformed sequence
    """
    rng = _make_rng(seed)

    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
    colors = ["red", "blue", "green", "yellow"]

    if difficulty == 1:
        # Simple 1-to-1 substitution
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            return [mapping.get(s, s) for s in seq]

    elif difficulty == 2:
        # Context-dependent: pairs matter
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            result = []
            i = 0
            while i < len(seq):
                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                    result.extend([pair_rule[2], pair_rule[2]])
                    i += 2
                else:
                    result.append(mapping.get(seq[i], seq[i]))
                    i += 1
            return result

    else:  # difficulty == 3
        # Multi-pass with conditional rules
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]
        cond = src[2]  # If this symbol is present, apply extra rule
        extra_map = {src[3]: dst[3]}

        rules = [
            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()
        ]
        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")
        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")
        rules.append("All other symbols stay the same throughout")

        def apply_rules(seq):
            # Pass 1
            result = [mapping1.get(s, s) for s in seq]
            # Pass 2
            result = [mapping2.get(s, s) for s in result]
            # Conditional
            if cond in seq:  # Check original sequence
                result = [extra_map.get(s, s) for s in result]
            return result

    # Generate examples
    all_items = []
    for _ in range(25):
        length = rng.randint(3, 6)
        seq = [rng.choice(shapes[:5]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    # Deduplicate by input
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_examples = min(15, len(unique_items) - 5)
    examples = unique_items[:n_examples]
    test_items = unique_items[n_examples:n_examples + 5]

    return RuleSystem(
        name=f"SymbolTransform-{seed}",
        description="Apply symbol transformation rules to input sequences",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="symbol",
    )


def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a novel number system / arithmetic.

    Input: expression in the invented system
    Rules: how operators work
    Output: numeric result
    """
    rng = _make_rng(seed)

    op_names = ["grok", "flim", "zorp", "quex", "blix"]
    ops = rng.sample(op_names, 3)

    if difficulty == 1:
        # Two operators: basic arithmetic with twist
        a_op, b_op = ops[0], ops[1]
        a_fn = lambda x, y: x + y + 1  # "grok" = add and increment
        b_fn = lambda x, y: abs(x - y)  # "flim" = absolute difference
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: absolute difference of x and y",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    elif difficulty == 2:
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        a_fn = lambda x, y: x * 2 + y
        b_fn = lambda x, y: (x + y) % 10
        c_fn = lambda x, y: max(x, y) - min(x, y) + 1
        rules = [
            f"'{a_op}(x, y)' means: double x, then add y",
            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",
            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",
        ]
        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}

    else:  # difficulty == 3
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        # Nested operations
        a_fn = lambda x, y: x + y + 1
        b_fn = lambda x, y: x * y
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: multiply x and y",
            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    # Generate examples
    all_items = []
    for _ in range(20):
        if difficulty <= 2:
            op_name = rng.choice(list(op_map.keys()))
            x = rng.randint(1, 9)
            y = rng.randint(1, 9)
            result = op_map[op_name](x, y)
            expr = f"{op_name}({x}, {y})"
        else:
            # Allow nesting
            if rng.random() < 0.5:
                op_name = rng.choice(list(op_map.keys()))
                x = rng.randint(1, 9)
                y = rng.randint(1, 9)
                result = op_map[op_name](x, y)
                expr = f"{op_name}({x}, {y})"
            else:
                inner_op = rng.choice(list(op_map.keys()))
                outer_op = rng.choice(list(op_map.keys()))
                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)
                inner_result = op_map[inner_op](x, y)
                result = op_map[outer_op](inner_result, z)
                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"

        all_items.append({"input": expr, "output": str(result)})

    # Deduplicate
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_ex = min(12, len(unique_items) - 5)
    examples = unique_items[:n_ex]
    test_items = unique_items[n_ex:n_ex + 5]

    return RuleSystem(
        name=f"NumberSystem-{seed}",
        description="Evaluate expressions using novel arithmetic operators",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="number",
    )


# Pre-generated systems for the benchmark
LEARNING_CURVE_SYSTEMS = [
    generate_symbol_system("lc_sym_easy", difficulty=1),
    generate_symbol_system("lc_sym_med", difficulty=2),
    generate_symbol_system("lc_sym_hard", difficulty=3),
    generate_number_system("lc_num_easy", difficulty=1),
    generate_number_system("lc_num_med", difficulty=2),
    generate_number_system("lc_num_hard", difficulty=3),
    generate_symbol_system("lc_sym_extreme1", difficulty=3),
    generate_number_system("lc_num_extreme2", difficulty=3),
]

# Systems for transfer testing
TRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)
TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)
TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)

# Systems for interference testing
INTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)
INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)


# ── Positional rule system (rules depend on position) ───────────────
def generate_positional_system(seed: str = "pos_default", difficulty: int = 3) -> RuleSystem:
    """
    Generate a positional rule system where transformations depend on
    element position in the sequence, not just identity.
    """
    rng = _make_rng(seed)
    shapes = ["△", "○", "□", "◇", "★", "⬡"]

    # Rules: position-dependent transformations
    pos_rules = [
        (0, shapes[0], shapes[4]),  # At position 0: △ → ★
        (1, shapes[1], shapes[5]),  # At position 1: ○ → ⬡
    ]
    swap_pair = (shapes[2], shapes[3])  # □ ↔ ◇ at even positions

    rules = [
        f"At position 0 (first element): replace {shapes[0]} with {shapes[4]}",
        f"At position 1 (second element): replace {shapes[1]} with {shapes[5]}",
        f"At even positions (0, 2, 4, ...): swap {shapes[2]} and {shapes[3]}",
        f"At odd positions (1, 3, 5, ...): duplicate the symbol (e.g., △ → △ △)",
        "Position-specific rules override the odd-position duplication rule",
    ]

    def apply_rules(seq):
        result = []
        for i, s in enumerate(seq):
            applied = False
            for pos, src, dst in pos_rules:
                if i == pos and s == src:
                    result.append(dst)
                    applied = True
                    break
            if not applied:
                if i % 2 == 0:  # even position
                    if s == swap_pair[0]:
                        result.append(swap_pair[1])
                    elif s == swap_pair[1]:
                        result.append(swap_pair[0])
                    else:
                        result.append(s)
                else:  # odd position - duplicate
                    for pos2, src2, _ in pos_rules:
                        if i == pos2 and s == src2:
                            applied = True
                            break
                    if not applied:
                        result.extend([s, s])
                    else:
                        result.append(s)
        return result

    all_items = []
    for _ in range(25):
        length = rng.randint(3, 5)
        seq = [rng.choice(shapes[:4]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    seen = set()
    unique = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique.append(item)

    rng.shuffle(unique)
    n_ex = min(12, len(unique) - 5)
    return RuleSystem(
        name=f"PositionalTransform-{seed}",
        description="Apply position-dependent transformation rules to symbol sequences",
        rules=rules,
        examples=unique[:n_ex],
        test_items=unique[n_ex:n_ex + 5],
        difficulty=difficulty,
        n_rules=len(rules),
        domain="positional",
    )


# ── Stateful accumulator system ─────────────────────────────────────
def generate_stateful_system(seed: str = "state_default", difficulty: int = 3) -> RuleSystem:
    """
    Generate a stateful system where output depends on running state
    accumulated through the sequence.
    """
    rng = _make_rng(seed)
    tokens = ["A", "B", "C", "D"]

    # State machine: counter starts at 0, each token modifies it
    token_effects = {
        "A": +2,
        "B": -1,
        "C": lambda s: s * 2 if s > 0 else 1,  # double if positive, else set to 1
        "D": 0,  # reset to 0
    }

    rules = [
        "Start with counter = 0",
        "A: add 2 to counter",
        "B: subtract 1 from counter",
        "C: if counter > 0, double it; otherwise set counter to 1",
        "D: reset counter to 0",
        "Output: the final counter value after processing all tokens left to right",
    ]

    def apply_rules(seq):
        counter = 0
        for t in seq:
            if t == "A":
                counter += 2
            elif t == "B":
                counter -= 1
            elif t == "C":
                counter = counter * 2 if counter > 0 else 1
            elif t == "D":
                counter = 0
        return str(counter)

    all_items = []
    for _ in range(30):
        length = rng.randint(3, 7)
        seq = [rng.choice(tokens) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": output})

    seen = set()
    unique = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique.append(item)

    rng.shuffle(unique)
    n_ex = min(12, len(unique) - 5)
    return RuleSystem(
        name=f"StatefulAccumulator-{seed}",
        description="Process token sequences through a stateful counter to compute final value",
        rules=rules,
        examples=unique[:n_ex],
        test_items=unique[n_ex:n_ex + 5],
        difficulty=difficulty,
        n_rules=len(rules),
        domain="stateful",
    )


# ── Far-transfer: genuine structural transfer ───────────────────────
def generate_structural_transfer(seed: str, base_system: RuleSystem) -> RuleSystem:
    """
    Generate a far-transfer system with genuinely different representation.

    For symbol systems: encode symbols as coordinate pairs, requiring the
    model to map coordinates → symbols → apply rules → symbols → coordinates.

    For number systems: encode as word-problem format with no operator syntax,
    requiring the model to identify which operator applies from context.
    """
    rng = _make_rng(seed)

    if base_system.domain == "symbol":
        # Map each shape to a coordinate pair
        shapes_all = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
        coords = [(i, j) for i in range(1, 4) for j in range(1, 4)]  # 9 coords
        rng.shuffle(coords)
        shape_to_coord = {}
        coord_to_shape = {}
        for i, s in enumerate(shapes_all[:len(coords)]):
            c = coords[i]
            shape_to_coord[s] = c
            coord_to_shape[c] = s

        def encode_seq(text):
            tokens = text.split()
            encoded = []
            for t in tokens:
                if t in shape_to_coord:
                    c = shape_to_coord[t]
                    encoded.append(f"({c[0]},{c[1]})")
                else:
                    encoded.append(t)
            return " ".join(encoded)

        # The transfer system has NO rules listed — just the coordinate mapping
        # and 2 worked examples. Model must figure out the structure.
        coord_legend = [f"({c[0]},{c[1]}) = {s}" for s, c in shape_to_coord.items()
                        if s in " ".join(e["input"] for e in base_system.examples + base_system.test_items)]

        return RuleSystem(
            name=f"CoordinateTransfer-{seed}",
            description=(
                "Same transformation rules as the base system, but symbols are encoded "
                "as coordinate pairs. Decode coordinates, apply rules, re-encode output."
            ),
            rules=[f"Coordinate mapping: {', '.join(coord_legend[:6])}",
                   "Apply the SAME transformation rules from the base system",
                   "Output the result as coordinate pairs"],
            examples=[{"input": encode_seq(e["input"]), "output": encode_seq(e["output"])}
                      for e in base_system.examples[:2]],  # Only 2 examples!
            test_items=[{"input": encode_seq(t["input"]), "output": encode_seq(t["output"])}
                        for t in base_system.test_items],
            difficulty=base_system.difficulty + 1,
            n_rules=3,
            domain="coordinate_transfer",
        )
    else:
        # Number system → word problem format
        # Extract operators from base system
        contexts = [
            "In a factory, workers {op} {x} units from line A with {y} units from line B. How many total units?",
            "A recipe calls for {op}-processing {x} grams of ingredient X and {y} grams of ingredient Y. What is the result?",
            "In the game, player scores are combined by {op}: first score is {x}, second score is {y}. Final score?",
        ]
        rng.shuffle(contexts)

        # Use base examples but reformat as word problems
        transfer_examples = []
        transfer_tests = []

        for item in base_system.examples[:2]:
            transfer_examples.append({
                "input": f"Word problem: {item['input']} (evaluate using the learned rules)",
                "output": item["output"],
            })

        for item in base_system.test_items:
            transfer_tests.append({
                "input": f"Word problem: {item['input']} (evaluate using the learned rules)",
                "output": item["output"],
            })

        return RuleSystem(
            name=f"ContextualTransfer-{seed}",
            description="Same arithmetic rules, but expressions are embedded in word-problem context",
            rules=["Apply the SAME operator rules you learned from the base system",
                   "Extract the expression from the word problem and evaluate"],
            examples=transfer_examples,
            test_items=transfer_tests,
            difficulty=base_system.difficulty + 1,
            n_rules=2,
            domain="contextual_transfer",
        )


FAR_TRANSFER_PAIRS = [
    {"base": generate_symbol_system("ft_sym_1", difficulty=2), "transfer": None},
    {"base": generate_symbol_system("ft_sym_2", difficulty=3), "transfer": None},
    {"base": generate_number_system("ft_num_1", difficulty=2), "transfer": None},
    {"base": generate_number_system("ft_num_2", difficulty=3), "transfer": None},
]
for pair in FAR_TRANSFER_PAIRS:
    pair["transfer"] = generate_structural_transfer(f"xfer_{pair['base'].name}", pair["base"])

# Hard condition systems — reduced training window (only 3 examples)
# Includes novel rule types: positional and stateful systems
HARD_LEARNING_SYSTEMS = [
    generate_symbol_system("lc_hard_sym_steep", difficulty=3),
    generate_number_system("lc_hard_num_steep", difficulty=3),
    generate_positional_system("lc_hard_positional", difficulty=3),
    generate_stateful_system("lc_hard_stateful", difficulty=3),
]


# ── New generators for v3 transfer / v4 interference ────────────────

def generate_incomplete_system(system: RuleSystem, n_omit: int = 1) -> RuleSystem:
    """
    Return a copy of a system with n_omit transformation rules removed from the rules list.

    The apply function (and therefore test_items answers) remains correct.
    The model must infer the missing rules from structural context.

    Omission strategy: skip rules that are transformation rules (not the
    catch-all "All other symbols stay the same" rule).
    """
    # Identify omittable indices: transformation rules, not catch-all
    omittable = [
        i for i, r in enumerate(system.rules)
        if not r.lower().startswith("all other")
        and not r.lower().startswith("output:")
        and not r.lower().startswith("start with")
    ]
    # Omit from the middle to preserve first and last context
    omit_idxs = set(omittable[1:1 + n_omit]) if len(omittable) > 1 else set(omittable[:n_omit])

    new_rules = [r for i, r in enumerate(system.rules) if i not in omit_idxs]
    new_system = RuleSystem(
        name=system.name + "-incomplete",
        description=system.description,
        rules=new_rules,
        examples=list(system.examples),   # full examples kept for context
        test_items=list(system.test_items),  # answers still valid
        difficulty=system.difficulty,
        n_rules=len(new_rules),
        domain=system.domain,
    )
    return new_system


def generate_zero_shot_transfer_system(seed: str = "zs_default") -> RuleSystem:
    """
    Generate a zero-shot structural transfer system.

    Uses the stateful accumulator representation — completely different from
    symbol/number systems. Only 1 worked example is provided in the benchmark;
    the model must infer the rules from description + structural analogy.
    """
    return generate_stateful_system(seed=seed, difficulty=3)


# ── v3 Transfer systems ──────────────────────────────────────────────

# Training system: symbol difficulty=2 (rules fully given — baseline)
TRANSFER_TRAIN_V3 = generate_symbol_system("v3_transfer_train", difficulty=2)

# Near transfer: same domain (symbol), difficulty=2 — 1 rule omitted
_NEAR_FULL_V3 = generate_symbol_system("v3_transfer_near", difficulty=2)
TRANSFER_NEAR_V3 = generate_incomplete_system(_NEAR_FULL_V3, n_omit=1)

# Far transfer: number domain, difficulty=2 — only 2 worked examples shown
TRANSFER_FAR_V3 = generate_number_system("v3_transfer_far", difficulty=2)

# Zero-shot structural: stateful system — only description + 1 example shown
TRANSFER_ZERO_SHOT_V3 = generate_zero_shot_transfer_system("v3_transfer_zeroshot")


# ── v4 Interference systems ──────────────────────────────────────────

# Easy tier: difficulty=1, 1 distractor (unchanged from v3)
INTERF_EASY_TARGET_V4 = generate_symbol_system("v4_easy_target", difficulty=1)
INTERF_EASY_DISTRACT_V4 = generate_symbol_system("v4_easy_distract", difficulty=1)

# Medium tier: difficulty=2, cross-contamination (overlapping symbol pool, different rules)
INTERF_MED_TARGET_V4 = generate_symbol_system("v4_med_target", difficulty=2)
INTERF_MED_DISTRACT_V4 = generate_symbol_system("v4_med_distract", difficulty=2)

# Hard tier: difficulty=3, 3 distractors, delayed interference
INTERF_HARD_TARGET_V4 = generate_symbol_system("v4_hard_target", difficulty=3)
INTERF_HARD_DIST1_V4 = generate_symbol_system("v4_hard_dist1", difficulty=3)
INTERF_HARD_DIST2_V4 = generate_symbol_system("v4_hard_dist2", difficulty=3)
INTERF_HARD_DIST3_V4 = generate_symbol_system("v4_hard_dist3", difficulty=3)
INTERF_HARD_FILLER_V4 = generate_symbol_system("v4_hard_filler", difficulty=2)  # filler for delay

# Extreme tier: 4 systems all difficulty=3, target gets only 2 examples
INTERF_EXT_TARGET_V4 = generate_symbol_system("v4_ext_target", difficulty=3)
INTERF_EXT_DIST1_V4 = generate_symbol_system("v4_ext_dist1", difficulty=3)
INTERF_EXT_DIST2_V4 = generate_symbol_system("v4_ext_dist2", difficulty=3)
INTERF_EXT_DIST3_V4 = generate_symbol_system("v4_ext_dist3", difficulty=3)


# ── v5 Interference: Rule Induction Under Interference ───────────────

def generate_similar_systems(seed: str, n_systems: int, difficulty: int, overlap_pct: float = 0.0) -> list[RuleSystem]:
    """
    Generate N symbol systems that share the SAME input symbol pool but have
    different rules.  overlap_pct controls how many rules are identical across
    systems (0.0 = all different, ~0.67 = 2/3 shared for 3 rules).

    Each system is generated with a unique sub-seed so rules differ.
    For overlap: the first system is generated normally; subsequent systems
    copy some rules from system-0 and regenerate the rest.
    """
    base_rng = _make_rng(seed)
    sub_seeds = [f"{seed}_sys{i}_{base_rng.randint(0, 999999)}" for i in range(n_systems)]

    systems = []
    for idx, ss in enumerate(sub_seeds):
        sys = generate_symbol_system(ss, difficulty=difficulty)
        systems.append(sys)

    return systems


def generate_shared_input_systems(
    seed: str, n_systems: int, n_shared_inputs: int, difficulty: int
) -> tuple[list[RuleSystem], list[list[str]]]:
    """
    Generate n_systems symbol systems + n_shared_inputs shared input sequences.
    Returns (systems, shared_inputs) where shared_inputs[i] is a list of symbol strings.

    For each shared input, each system produces a potentially different output.
    We verify that the combination of outputs across shared inputs uniquely
    identifies each system (needed for Tier 4 query-pair matching).
    """
    base_rng = _make_rng(seed)
    shapes = ["△", "○", "□", "◇", "★"]

    systems = []
    sub_seeds = [f"{seed}_shared_sys{i}_{base_rng.randint(0, 999999)}" for i in range(n_systems)]
    for ss in sub_seeds:
        systems.append(generate_symbol_system(ss, difficulty=difficulty))

    # Generate shared inputs — each must produce UNIQUE outputs across all systems
    shared_inputs = []
    max_attempts = 200
    attempt = 0
    while len(shared_inputs) < n_shared_inputs and attempt < max_attempts:
        attempt += 1
        length = base_rng.randint(3, 5)
        seq = [base_rng.choice(shapes) for _ in range(length)]
        # Check all systems produce unique outputs on this input
        outputs = []
        for sys in systems:
            out = _apply_system_to_seq(sys, seq)
            outputs.append(" ".join(out))
        if len(set(outputs)) == len(systems):
            shared_inputs.append(seq)

    if len(shared_inputs) < n_shared_inputs:
        raise ValueError(f"Could not find {n_shared_inputs} shared inputs with unique outputs after {max_attempts} attempts (seed={seed})")

    return systems, shared_inputs


def _apply_system_to_seq(system: RuleSystem, seq: list[str]) -> list[str]:
    """
    Re-derive the apply_rules function for a system by regenerating it
    with the same seed and difficulty, then applying to the given sequence.
    """
    # We regenerate to get the closure. The system's name encodes the seed.
    # Extract seed from name: "SymbolTransform-{seed}"
    seed = system.name.replace("SymbolTransform-", "")
    rebuilt = generate_symbol_system(seed, difficulty=system.difficulty)
    # Use the rebuilt system's internal apply by running through examples to verify,
    # then apply to our sequence.
    # Actually we need the closure directly. Let's rebuild:
    rng = _make_rng(seed)
    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]

    if system.difficulty == 1:
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        return [mapping.get(s, s) for s in seq]

    elif system.difficulty == 2:
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])
        result = []
        i = 0
        while i < len(seq):
            if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                result.extend([pair_rule[2], pair_rule[2]])
                i += 2
            else:
                result.append(mapping.get(seq[i], seq[i]))
                i += 1
        return result

    else:  # difficulty == 3
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}
        cond = src[2]
        extra_map = {src[3]: dst[3]}
        result = [mapping1.get(s, s) for s in seq]
        result = [mapping2.get(s, s) for s in result]
        if cond in seq:
            result = [extra_map.get(s, s) for s in result]
        return result


# ── Pre-generated v5 systems ──────────────────────────────────────────

# Tier 1: 5 single clean-induction systems, difficulty=2
INTERF_V5_TIER1_SYSTEMS = [
    generate_symbol_system(f"v5_t1_{i}", difficulty=2) for i in range(5)
]

# Tier 2: 5 pairs of similar systems (A=target, B=distractor)
INTERF_V5_TIER2_PAIRS = []
for i in range(5):
    pair = generate_similar_systems(f"v5_t2_pair{i}", n_systems=2, difficulty=2)
    INTERF_V5_TIER2_PAIRS.append((pair[0], pair[1]))

# Tier 3: 5 triples of systems (α=target, β=primer, γ=distractor)
INTERF_V5_TIER3_TRIPLES = []
for i in range(5):
    triple = generate_similar_systems(f"v5_t3_triple{i}", n_systems=3, difficulty=2)
    INTERF_V5_TIER3_TRIPLES.append((triple[0], triple[1], triple[2]))

# Tier 4: 5 sets of 4 systems with shared inputs
INTERF_V5_TIER4_SETS = []
for i in range(5):
    systems, shared = generate_shared_input_systems(
        f"v5_t4_set{i}", n_systems=4, n_shared_inputs=3, difficulty=2
    )
    INTERF_V5_TIER4_SETS.append((systems, shared))


"""
Learning Benchmark 3: Rule Induction Under Interference (v5)

Core insight: never state rules as text. Force the model to INDUCE rules
from input→output examples. When induction competes with similar-looking
examples from different systems, interference becomes real.

Tier Structure:
- Tier 1 (0.10): Clean induction — 5 examples from ONE system, induce & apply
- Tier 2 (0.25): Labeled groups — 2 groups (A/B), 4 examples each, same symbol set
- Tier 3 (0.35): Interleaved + anti-pattern priming — 3 systems scattered, worked
                  example of WRONG system primes incorrect procedure
- Tier 4 (0.30): Unlabeled clustering — 4 systems, 12 unlabeled examples, query pair
                  identifies target system

Composite = 0.10 * tier1 + 0.25 * tier2 + 0.35 * tier3 + 0.30 * tier4
"""

import re
import json
import random
import hashlib


def _strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def normalize_output(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text


def check_output(model_output: str, expected: str) -> bool:
    m = normalize_output(model_output)
    e = normalize_output(expected)
    return e in m or m in e


def _extract_answer(raw: str) -> str:
    cleaned = _strip_think(raw)
    cleaned = re.sub(r'//.*', '', cleaned)
    try:
        parsed = json.loads(re.search(r'\{.*\}', cleaned, re.DOTALL).group())
        return str(parsed.get("answer", cleaned))
    except Exception:
        return cleaned


def _fmt_examples(system, n: int) -> str:
    lines = []
    for ex in system.examples[:n]:
        lines.append(f"{ex['input']} → {ex['output']}")
    return "\n".join(lines)


def _make_rng(seed: str) -> random.Random:
    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)
    return random.Random(h)


# ── Tier 1: Clean Induction ─────────────────────────────────────────

def run_tier1(llm) -> float:
    correct = 0
    total = 0
    for si, system in enumerate(INTERF_V5_TIER1_SYSTEMS):
        test_item = system.test_items[0]
        prompt = (
            "Study these transformations:\n"
            + _fmt_examples(system, 5)
            + f"\n\nApplying the same pattern:\n{test_item['input']} → ?\n\n"
            + 'Respond with ONLY: {"answer": "<output>"}'
        )
        with kbench.chats.new(f"t1_{si}"):
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            if check_output(answer, test_item["output"]):
                correct += 1
            total += 1
    return correct / total if total else 0


# ── Tier 2: Labeled Groups ──────────────────────────────────────────

def run_tier2(llm) -> float:
    correct = 0
    total = 0
    for pi, (target, distractor) in enumerate(INTERF_V5_TIER2_PAIRS):
        test_item = target.test_items[0]
        prompt = (
            "Two transformation systems are shown below.\n\n"
            "--- Group A ---\n"
            + _fmt_examples(target, 4)
            + "\n\n--- Group B ---\n"
            + _fmt_examples(distractor, 4)
            + f"\n\nApplying the GROUP A pattern:\n{test_item['input']} → ?\n\n"
            + 'Respond with ONLY: {"answer": "<output>"}'
        )
        with kbench.chats.new(f"t2_{pi}"):
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            if check_output(answer, test_item["output"]):
                correct += 1
            total += 1
    return correct / total if total else 0


# ── Tier 3: Interleaved + Anti-Pattern Priming ──────────────────────

def run_tier3(llm) -> float:
    correct = 0
    total = 0
    for ti, (alpha, beta, gamma) in enumerate(INTERF_V5_TIER3_TRIPLES):
        rng = _make_rng(f"t3_shuffle_{ti}")

        # Build interleaved examples: 3 per system, scattered
        tagged = []
        for ex in alpha.examples[:3]:
            tagged.append(f"[α] {ex['input']} → {ex['output']}")
        for ex in beta.examples[:3]:
            tagged.append(f"[β] {ex['input']} → {ex['output']}")
        for ex in gamma.examples[:3]:
            tagged.append(f"[γ] {ex['input']} → {ex['output']}")
        rng.shuffle(tagged)

        # Build worked example of β (the WRONG system) as primer
        beta_test = beta.test_items[0]
        worked = (
            f"\nHere is a worked example of [β]:\n"
            f"Input: {beta_test['input']}\n"
            f"Step 1: Apply first transformation rule\n"
            f"Step 2: Apply second transformation rule\n"
            f"Output: {beta_test['output']}\n"
        )

        test_item = alpha.test_items[0]
        prompt = (
            "Observe these transformations from three systems:\n\n"
            + "\n".join(tagged)
            + worked
            + f"\nNow apply the [α] pattern:\n{test_item['input']} → ?\n\n"
            + 'Respond with ONLY: {"answer": "<output>"}'
        )
        with kbench.chats.new(f"t3_{ti}"):
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            if check_output(answer, test_item["output"]):
                correct += 1
            total += 1
    return correct / total if total else 0


# ── Tier 4: Unlabeled Clustering ────────────────────────────────────

def run_tier4(llm) -> float:
    correct = 0
    total = 0
    for si, (systems, shared_inputs) in enumerate(INTERF_V5_TIER4_SETS):
        rng = _make_rng(f"t4_shuffle_{si}")

        # Compute outputs for each system on each shared input
        all_transformations = []  # (input_str, output_str, system_idx)
        outputs_by_system = {}  # system_idx -> list of output strings
        for sys_idx, sys in enumerate(systems):
            outputs_by_system[sys_idx] = []
            for seq in shared_inputs:
                out = _apply_system_to_seq(sys, seq)
                inp_str = " ".join(seq)
                out_str = " ".join(out)
                all_transformations.append((inp_str, out_str, sys_idx))
                outputs_by_system[sys_idx].append(out_str)

        # Verify uniqueness: each system should have unique output signature
        sigs = {}
        for sys_idx, outs in outputs_by_system.items():
            sig = tuple(outs)
            if sig in sigs:
                # Collision — skip this set
                continue
            sigs[sig] = sys_idx

        if len(sigs) < len(systems):
            # Can't disambiguate — skip
            continue

        # Pick target system (first one)
        target_idx = 0
        target_sys = systems[target_idx]

        # Query pair: use a shared input to identify the target
        query_inp = " ".join(shared_inputs[0])
        query_out = outputs_by_system[target_idx][0]

        # Verify query pair uniquely identifies target
        matching = [idx for idx, outs in outputs_by_system.items() if outs[0] == query_out]
        if len(matching) != 1:
            continue  # ambiguous, skip

        # Build unlabeled block (shuffled)
        lines = []
        for inp_str, out_str, _ in all_transformations:
            lines.append(f"{inp_str} → {out_str}")
        rng.shuffle(lines)

        # Test item: use a test item from the target system
        test_item = target_sys.test_items[0]

        prompt = (
            f"{'Twelve' if len(all_transformations) == 12 else str(len(all_transformations))} "
            f"transformations are shown (from several different systems, unlabeled):\n\n"
            + "\n".join(lines)
            + f"\n\nThis transformation follows one of the patterns above:\n"
            + f"{query_inp} → {query_out}\n\n"
            + f"Applying that SAME pattern:\n{test_item['input']} → ?\n\n"
            + 'Respond with ONLY: {"answer": "<output>"}'
        )
        with kbench.chats.new(f"t4_{si}"):
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            if check_output(answer, test_item["output"]):
                correct += 1
            total += 1
    return correct / total if total else 0


@kbench.task(name="Rule Induction Under Interference")
def learning_interference(llm) -> float:
    """Rule Induction Under Interference Benchmark (v5).

    Four tiers testing rule induction with increasing interference:
    - Tier 1 (0.10): Clean induction from examples
    - Tier 2 (0.25): Labeled groups with overlapping symbols
    """

    print("\n" + "=" * 60)
    print("RULE INDUCTION UNDER INTERFERENCE v5")
    print("=" * 60)

    # Tier 1
    print("\n--- TIER 1: Clean Induction (5 items) ---")
    t1 = run_tier1(llm)
    print(f"  Accuracy: {t1:.1%}")

    # Tier 2
    print("\n--- TIER 2: Labeled Groups (5 items) ---")
    t2 = run_tier2(llm)
    print(f"  Accuracy: {t2:.1%}")

    # Tier 3
    print("\n--- TIER 3: Interleaved + Priming (5 items) ---")
    t3 = run_tier3(llm)
    print(f"  Accuracy: {t3:.1%}")

    # Tier 4
    print("\n--- TIER 4: Unlabeled Clustering (up to 5 items) ---")
    t4 = run_tier4(llm)
    print(f"  Accuracy: {t4:.1%}")

    # Composite
    score = round(0.10 * t1 + 0.25 * t2 + 0.35 * t3 + 0.30 * t4, 4)
    score = max(0.0, min(1.0, score))

    print(f"\n{'=' * 60}")
    print(f"COMPOSITE SCORE: {score:.4f}")
    print(f"  Tier 1: {t1:.4f} × 0.10 = {0.10 * t1:.4f}")
    print(f"  Tier 2: {t2:.4f} × 0.25 = {0.25 * t2:.4f}")
    print(f"  Tier 3: {t3:.4f} × 0.35 = {0.35 * t3:.4f}")
    print(f"  Tier 4: {t4:.4f} × 0.30 = {0.30 * t4:.4f}")
    print(f"{'=' * 60}")

    return score


In [ ]:
learning_interference.run(llm=kbench.llm)